In [2]:


# ──────────────────────────────────────────────────────────────────────────────
import os, warnings, shutil, math, random
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import tensorflow as tf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


if not os.path.exists("content/Skin-Disease"):
    shutil.copytree("drive/MyDrive/Dataset/Skin-Disease",
                    "content/Skin-Disease")
print(f"TensorFlow {tf.__version__}")

# ──────────────────────────────────────────────────────────────────────────────
#  GPU / MIXED-PRECISION
# ──────────────────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs: {len(gpus)}")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
policy = "mixed_float16" if gpus else "float32"
tf.keras.mixed_precision.set_global_policy(policy)
print(f"Compute dtype: {tf.keras.mixed_precision.global_policy().compute_dtype}")

# ──────────────────────────────────────────────────────────────────────────────
#  CONFIG
# ──────────────────────────────────────────────────────────────────────────────
# ── Colab: set DATA_ROOT to your local copy (not Drive path) ──────────────────
# from google.colab import drive; drive.mount("/content/drive")
# if not os.path.exists("/content/Skin-Disease"):
#     shutil.copytree("/content/drive/MyDrive/Dataset/Skin-Disease",
#                     "/content/Skin-Disease")
DATA_ROOT    = "content/Skin-Disease"         # ← change if needed
RESPLIT_ROOT = "data_resplit"         # new balanced splits go here

IMG_SIZE     = (260, 260)             # larger = better texture detail
BATCH_SIZE   = 32
SEED         = 42
AUTOTUNE     = tf.data.AUTOTUNE

EPOCHS_WU    = 12    # phase 1: head only
EPOCHS_FT1   = 15    # phase 2: top-30 layers
EPOCHS_FT2   = 20    # phase 3: top-80 layers (very low LR)

LR_WU        = 3e-3
LR_FT1       = 5e-5
LR_FT2       = 8e-6
DROPOUT      = 0.45
LABEL_SMOOTH = 0.10
MODEL_PATH   = "skin_disease_resnet50v2.keras"
TTA_STEPS    = 8     # number of augmented copies for TTA

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)



# ──────────────────────────────────────────────────────────────────────────────
#  STEP 2 — STRATIFIED RESPLIT  (pool all data → 70/15/15)
# ──────────────────────────────────────────────────────────────────────────────
def collect_all_images(root):
    """Returns {class_name: [abs_path, ...]} from all splits."""
    data = {}
    for split in ("train", "valid", "test"):
        sd = os.path.join(root, split)
        if not os.path.isdir(sd): continue
        for cls in os.listdir(sd):
            cp = os.path.join(sd, cls)
            if not os.path.isdir(cp): continue
            imgs = [os.path.join(cp, f) for f in os.listdir(cp)
                    if f.lower().endswith((".jpg", ".jpeg", ".png"))]
            data.setdefault(cls, []).extend(imgs)
    return data

def stratified_resplit(src_root, dst_root, train_r=0.70, valid_r=0.15,
                       seed=SEED, min_test=8):
    """Pool all images and re-split with stratification."""
    if os.path.exists(dst_root):
        print(f"  Re-split already exists at '{dst_root}', skipping.")
        return

    all_data = collect_all_images(src_root)
    class_names = sorted(all_data.keys())
    print(f"\n══ Stratified Resplit → {dst_root} ══")
    print(f"  {'Class':<22}  {'Total':>6}  {'Train':>6}  {'Valid':>6}  {'Test':>6}")
    print("  " + "─" * 50)

    for cls in class_names:
        imgs   = all_data[cls]
        random.shuffle(imgs)
        n      = len(imgs)
        n_test = max(min_test, int(n * (1 - train_r - valid_r)))
        n_val  = max(min_test, int(n * valid_r))
        n_tr   = n - n_test - n_val

        splits_map = (
            ("train", imgs[:n_tr]),
            ("valid", imgs[n_tr:n_tr + n_val]),
            ("test",  imgs[n_tr + n_val:]),
        )
        print(f"  {cls:<22}  {n:>6}  {n_tr:>6}  {n_val:>6}  {len(imgs[n_tr+n_val:]):>6}")

        for split_name, paths in splits_map:
            out_dir = os.path.join(dst_root, split_name, cls)
            os.makedirs(out_dir, exist_ok=True)
            for p in paths:
                dst = os.path.join(out_dir, Path(p).name)
                if not os.path.exists(dst):
                    shutil.copy2(p, dst)

stratified_resplit(DATA_ROOT, RESPLIT_ROOT)

TRAIN_DIR = os.path.join(RESPLIT_ROOT, "train")
VALID_DIR  = os.path.join(RESPLIT_ROOT, "valid")
TEST_DIR   = os.path.join(RESPLIT_ROOT, "test")

def count_images(directory):
    counts = {}
    for cls in sorted(os.listdir(directory)):
        p = os.path.join(directory, cls)
        if os.path.isdir(p):
            n = len([f for f in os.listdir(p)
                     if f.lower().endswith((".jpg", ".jpeg", ".png"))])
            if n: counts[cls] = n
    return counts

train_counts = count_images(TRAIN_DIR)
class_names  = sorted(train_counts.keys())
num_classes  = len(class_names)
print(f"\nClasses ({num_classes}): {class_names}")

# ──────────────────────────────────────────────────────────────────────────────
#  STEP 3 — CLASS WEIGHTS
# ──────────────────────────────────────────────────────────────────────────────
total_train   = sum(train_counts.values())
class_weights = {
    i: total_train / (num_classes * train_counts[cls])
    for i, cls in enumerate(class_names)
}
print("\n══ Class weights ══")
for i, cls in enumerate(class_names):
    print(f"  {cls:<22}  {class_weights[i]:.3f}")

# ──────────────────────────────────────────────────────────────────────────────
#  STEP 4 — PREPROCESSING  (ResNet50V2 expects [-1, 1])
# ──────────────────────────────────────────────────────────────────────────────
resnet_preprocess = tf.keras.applications.resnet_v2.preprocess_input

# ──────────────────────────────────────────────────────────────────────────────
#  STEP 5 — AUGMENTATION  (pure tf.image ops — safe inside ds.map)
# ──────────────────────────────────────────────────────────────────────────────
def random_rotate_tf(image, max_deg=20.0):
    angle = tf.random.uniform([], -max_deg, max_deg) * (math.pi / 180.0)
    ca, sa = tf.math.cos(angle), tf.math.sin(angle)
    h = tf.cast(tf.shape(image)[0], tf.float32)
    w = tf.cast(tf.shape(image)[1], tf.float32)
    cx, cy = w / 2.0, h / 2.0
    t = [ca, -sa, cx - cx*ca + cy*sa,
         sa,  ca, cy - cx*sa - cy*ca,
         0.0, 0.0]
    t = tf.reshape(tf.stack(t), [1, 8])
    image = tf.cast(image, tf.float32)
    image = tf.raw_ops.ImageProjectiveTransformV3(
        images=tf.expand_dims(image, 0), transforms=t,
        output_shape=tf.shape(image)[:2],
        interpolation="BILINEAR", fill_mode="REFLECT", fill_value=0.0)[0]
    return image

def random_zoom_tf(image, lo=0.85, hi=1.15):
    """Zoom in/out by cropping then resizing back."""
    factor = tf.random.uniform([], lo, hi)
    h = tf.cast(tf.shape(image)[0], tf.int32)
    w = tf.cast(tf.shape(image)[1], tf.int32)
    new_h = tf.maximum(tf.cast(tf.cast(h, tf.float32) / factor, tf.int32), 1)
    new_w = tf.maximum(tf.cast(tf.cast(w, tf.float32) / factor, tf.int32), 1)
    image = tf.image.resize_with_crop_or_pad(tf.cast(image, tf.float32), new_h, new_w)
    image = tf.image.resize_with_crop_or_pad(image, h, w)
    return image

def augment_fn(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, 0.30)
    image = tf.image.random_contrast(image, 0.70, 1.30)
    image = tf.image.random_saturation(image, 0.60, 1.40)
    image = tf.image.random_hue(image, 0.06)
    image = random_rotate_tf(image, max_deg=25.0)
    image = random_zoom_tf(image, lo=0.85, hi=1.15)
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, label

# ──────────────────────────────────────────────────────────────────────────────
#  STEP 6 — MIXUP  (reduces inter-class confusion for visually similar classes)
# ──────────────────────────────────────────────────────────────────────────────
def mixup_batch(images, labels, alpha=0.3):
    """Apply Mixup to a batch: interpolates pairs of images and labels."""
    batch_size = tf.shape(images)[0]
    lam = tf.random.uniform([batch_size], 0.0, 1.0)
    # Only apply mixup to a fraction — blend with original
    lam = tf.maximum(lam, 1.0 - lam)   # keep dominant class ≥ 0.5
    idx = tf.random.shuffle(tf.range(batch_size))
    images2 = tf.gather(images, idx)
    labels2 = tf.gather(labels, idx)
    lam_img = tf.reshape(lam, [batch_size, 1, 1, 1])
    lam_lbl = tf.reshape(lam, [batch_size, 1])
    mixed_images = lam_img * images + (1.0 - lam_img) * images2
    mixed_labels = lam_lbl * labels + (1.0 - lam_lbl) * labels2
    return mixed_images, mixed_labels

# ──────────────────────────────────────────────────────────────────────────────
#  STEP 7 — FOCAL LOSS
# ──────────────────────────────────────────────────────────────────────────────
class FocalLoss(tf.keras.losses.Loss):
    """
    Focal Loss: down-weights easy examples, focuses on hard ones.
    Better than cross-entropy for imbalanced multi-class problems.
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    """
    def __init__(self, gamma=2.0, label_smoothing=0.1, **kwargs):
        super().__init__(**kwargs)
        self.gamma          = gamma
        self.label_smoothing = label_smoothing

    def call(self, y_true, y_pred):
        # Apply label smoothing
        n_cls   = tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_true  = y_true * (1.0 - self.label_smoothing) + \
                  self.label_smoothing / n_cls
        y_pred  = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce      = -y_true * tf.math.log(y_pred)
        weight  = tf.pow(1.0 - y_pred, self.gamma)
        focal   = weight * ce
        return tf.reduce_mean(tf.reduce_sum(focal, axis=-1))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"gamma": self.gamma,
                    "label_smoothing": self.label_smoothing})
        return cfg

# ──────────────────────────────────────────────────────────────────────────────
#  STEP 8 — tf.data PIPELINE
# ──────────────────────────────────────────────────────────────────────────────
def make_dataset(directory, augment=False, shuffle=True, mixup=False):
    ds = tf.keras.utils.image_dataset_from_directory(
        directory,
        image_size=IMG_SIZE,
        batch_size=None,
        label_mode="categorical",
        class_names=class_names,
        seed=SEED,
        shuffle=shuffle,
    )
    names = ds.class_names

    # Cache raw images in RAM — augmentation stays random each epoch
    ds = ds.cache()

    if shuffle:
        ds = ds.shuffle(buffer_size=2000, seed=SEED, reshuffle_each_iteration=True)

    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)

    def preprocess(img, lbl):
        img = tf.cast(img, tf.float32)
        img = resnet_preprocess(img)    # → [-1, 1]
        return img, lbl

    ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)

    if mixup:
        ds = ds.map(lambda x, y: mixup_batch(x, y, alpha=0.3),
                    num_parallel_calls=AUTOTUNE)

    ds = ds.prefetch(AUTOTUNE)
    return ds, names

print("\n── Building tf.data pipelines ──")
train_ds, _ = make_dataset(TRAIN_DIR, augment=True,  shuffle=True,  mixup=True)
valid_ds, _ = make_dataset(VALID_DIR, augment=False, shuffle=False, mixup=False)
test_ds,  _ = make_dataset(TEST_DIR,  augment=False, shuffle=False, mixup=False)

print(f"  Train batches : {len(train_ds)}")
print(f"  Valid batches : {len(valid_ds)}")
print(f"  Test  batches : {len(test_ds)}")

# ──────────────────────────────────────────────────────────────────────────────
#  STEP 9 — MODEL: ResNet50V2
# ──────────────────────────────────────────────────────────────────────────────
def build_resnet_model(num_classes):
    backbone = tf.keras.applications.ResNet50V2(
        weights="imagenet",
        include_top=False,
        input_shape=(*IMG_SIZE, 3),
    )
    backbone.trainable = False

    inp = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x   = backbone(inp, training=False)
    x   = tf.keras.layers.GlobalAveragePooling2D()(x)

    # Wider head for 11-class task
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(DROPOUT)(x)
    x = tf.keras.layers.Dense(
            512, activation="relu",
            kernel_regularizer=tf.keras.regularizers.l2(2e-4))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(DROPOUT * 0.6)(x)
    x = tf.keras.layers.Dense(
            256, activation="relu",
            kernel_regularizer=tf.keras.regularizers.l2(2e-4))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(DROPOUT * 0.4)(x)

    out = tf.keras.layers.Dense(num_classes, activation="softmax",
                                dtype="float32")(x)
    return tf.keras.Model(inp, out), backbone

model, backbone = build_resnet_model(num_classes)
model.summary(line_length=80, expand_nested=False)

# ──────────────────────────────────────────────────────────────────────────────
#  STEP 10 — CALLBACKS
# ──────────────────────────────────────────────────────────────────────────────
def make_callbacks(tag, patience_es=8, patience_lr=4):
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=patience_es,
            restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.4, patience=patience_lr,
            min_lr=1e-9, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=f"best_{tag}.keras",
            monitor="val_accuracy", save_best_only=True, verbose=0),
    ]

focal_loss = FocalLoss(gamma=2.0, label_smoothing=LABEL_SMOOTH)

# ──────────────────────────────────────────────────────────────────────────────
#  STEP 11 — PHASE 1: HEAD WARM-UP  (backbone 100% frozen)
# ──────────────────────────────────────────────────────────────────────────────
print("\n══ Phase 1: Head Warm-up ══")
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_WU),
    loss=focal_loss,
    metrics=["accuracy"],
)
h1 = model.fit(
    train_ds, validation_data=valid_ds,
    epochs=EPOCHS_WU,
    class_weight=class_weights,
    callbacks=make_callbacks("phase1"),
    verbose=1,
)

# ──────────────────────────────────────────────────────────────────────────────
#  STEP 12 — PHASE 2: FINE-TUNE TOP-30 LAYERS
# ──────────────────────────────────────────────────────────────────────────────
print("\n══ Phase 2: Fine-tune top-30 backbone layers ══")
backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False
print(f"  Unfrozen layers : {sum(1 for l in backbone.layers if l.trainable)}")

model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_FT1),
    loss=focal_loss,
    metrics=["accuracy"],
)
h2 = model.fit(
    train_ds, validation_data=valid_ds,
    epochs=EPOCHS_FT1,
    class_weight=class_weights,
    callbacks=make_callbacks("phase2"),
    verbose=1,
)

# ──────────────────────────────────────────────────────────────────────────────
#  STEP 13 — PHASE 3: DEEP FINE-TUNE TOP-80 LAYERS  (very low LR)
# ──────────────────────────────────────────────────────────────────────────────
print("\n══ Phase 3: Deep fine-tune top-80 backbone layers ══")
for layer in backbone.layers[:-80]:
    layer.trainable = False
for layer in backbone.layers[-80:]:
    layer.trainable = True
print(f"  Unfrozen layers : {sum(1 for l in backbone.layers if l.trainable)}")

steps_per_epoch = len(train_ds)
total_steps     = EPOCHS_FT2 * steps_per_epoch
cosine_lr = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=LR_FT2,
    decay_steps=total_steps,
    alpha=1e-8,
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(cosine_lr),
    loss=focal_loss,
    metrics=["accuracy"],
)
h3 = model.fit(
    train_ds, validation_data=valid_ds,
    epochs=EPOCHS_FT2,
    class_weight=class_weights,
    callbacks=make_callbacks("phase3", patience_es=10, patience_lr=5),
    verbose=1,
)

model.save(MODEL_PATH)
print(f"\nModel saved → {MODEL_PATH}")
from google.colab import drive
drive.mount('/content/drive')

import shutil
# Creates a permanent copy in your Drive dataset folder
shutil.copy(MODEL_PATH, "/content/drive/MyDrive/Dataset/skin_disease_resnet50v2.keras")
print(f"Model saved locally and backed up permanently to Google Drive!")
# ──────────────────────────────────────────────────────────────────────────────
#  STEP 14 — STANDARD EVALUATION
# ──────────────────────────────────────────────────────────────────────────────
print("\n═══ Standard Evaluation (no TTA) ═══")
loss, acc = model.evaluate(test_ds, verbose=1)
print(f"\n  Test Accuracy (standard) : {acc * 100:.2f}%")

y_pred_all, y_true_all = [], []
for bx, by in test_ds:
    probs = model.predict(bx, verbose=0)
    y_pred_all.extend(np.argmax(probs, axis=1))
    y_true_all.extend(np.argmax(by.numpy(), axis=1))
y_pred_all = np.array(y_pred_all)
y_true_all = np.array(y_true_all)

print("\n═══ Per-class Report ═══")
print(classification_report(y_true_all, y_pred_all,
                             target_names=class_names, digits=3))

# ──────────────────────────────────────────────────────────────────────────────
#  STEP 15 — TEST-TIME AUGMENTATION  (TTA: free +2-4% accuracy boost)
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n═══ TTA Evaluation ({TTA_STEPS} augmented passes) ═══")

def tta_predict(model, test_directory, n_augments=TTA_STEPS):
    """
    For each test image, predict n_augments times with augmentation,
    then average the softmax probabilities.
    """
    # Load raw images (no preprocessing yet)
    raw_ds = tf.keras.utils.image_dataset_from_directory(
        test_directory,
        image_size=IMG_SIZE,
        batch_size=None,
        label_mode="categorical",
        class_names=class_names,
        shuffle=False,
    )

    all_labels = []
    all_probs  = []

    for img, lbl in raw_ds:
        all_labels.append(np.argmax(lbl.numpy()))
        img_f = tf.cast(img, tf.float32)

        # Collect n_augments predictions
        aug_probs = []

        # Original (no augmentation)
        orig = resnet_preprocess(img_f)
        orig = tf.expand_dims(orig, 0)
        aug_probs.append(model.predict(orig, verbose=0)[0])

        # Augmented versions
        for _ in range(n_augments - 1):
            aug_img, _ = augment_fn(img_f, lbl)
            aug_img    = resnet_preprocess(aug_img)
            aug_img    = tf.expand_dims(aug_img, 0)
            aug_probs.append(model.predict(aug_img, verbose=0)[0])

        # Average probabilities
        avg_prob = np.mean(aug_probs, axis=0)
        all_probs.append(avg_prob)

    return np.array(all_labels), np.array(all_probs)

y_true_tta, probs_tta = tta_predict(model, TEST_DIR)
y_pred_tta = np.argmax(probs_tta, axis=1)

tta_acc = np.mean(y_pred_tta == y_true_tta)
print(f"  Test Accuracy (TTA)      : {tta_acc * 100:.2f}%")
print(f"  Improvement from TTA     : +{(tta_acc - acc) * 100:.2f}%")

print("\n═══ Per-class Report (with TTA) ═══")
print(classification_report(y_true_tta, y_pred_tta,
                             target_names=class_names, digits=3))



# ──────────────────────────────────────────────────────────────────────────────
#  STEP 18 — FIXED PREDICTION FUNCTION  (TTA-powered)
# ──────────────────────────────────────────────────────────────────────────────
def predict_image(image_path, model_path=MODEL_PATH, names=None,
                  use_tta=True, n_tta=TTA_STEPS, save_path="prediction_result.png"):
    """
    Predict skin condition from one image.
    Optionally uses TTA for more reliable confidence scores.

    Usage:
        predict_image("my_face.jpg")
        predict_image("my_face.jpg", use_tta=False)   # faster, no TTA
    """
    if names is None:
        names = class_names

    mdl    = tf.keras.models.load_model(model_path,
                 custom_objects={"FocalLoss": FocalLoss})
    img_pil = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    img_arr = tf.keras.utils.img_to_array(img_pil)   # [0, 255]
    img_f   = tf.cast(img_arr, tf.float32)

    if use_tta:
        aug_probs = []
        # Original
        orig = resnet_preprocess(img_f)
        aug_probs.append(mdl.predict(tf.expand_dims(orig, 0), verbose=0)[0])
        # Augmented
        dummy_label = tf.zeros(num_classes)
        for _ in range(n_tta - 1):
            aug_img, _ = augment_fn(img_f, dummy_label)
            aug_img    = resnet_preprocess(aug_img)
            aug_probs.append(mdl.predict(tf.expand_dims(aug_img, 0), verbose=0)[0])
        probs = np.mean(aug_probs, axis=0)
    else:
        inp   = resnet_preprocess(img_f)
        probs = mdl.predict(tf.expand_dims(inp, 0), verbose=0)[0]

    top5   = probs.argsort()[-5:][::-1]
    labels = [names[i] for i in top5]
    scores = probs[top5]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    ax1.imshow(img_pil); ax1.axis("off")
    ax1.set_title("Input Image", fontsize=11)

    colors = plt.cm.RdYlGn(np.linspace(0.25, 0.85, 5))
    bars   = ax2.barh(labels[::-1], scores[::-1] * 100, color=colors)
    ax2.set_xlabel("Confidence (%)"); ax2.set_xlim(0, 105)
    mode = f"TTA ×{n_tta}" if use_tta else "Standard"
    ax2.set_title(f"Top-5 Predictions ({mode})")
    for bar, s in zip(bars, scores[::-1]):
        ax2.text(bar.get_width() + 0.5,
                 bar.get_y() + bar.get_height() / 2,
                 f"{s*100:.1f}%", va="center", fontsize=9)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    method = f"TTA (×{n_tta})" if use_tta else "Standard"
    print(f"\n🔍  Image     : {Path(image_path).name}  [{method}]")
    print(f"   ✅  Predicted : {labels[0]}  ({scores[0]*100:.2f}%)")
    print("\n   Top-5:")
    for lb, sc in zip(labels, scores):
        print(f"     {lb:<22s} {sc*100:5.1f}%  {'█'*int(sc*40)}")

    return labels[0], float(scores[0])


# Usage:
# predict_image("path/to/your/face_photo.jpg")



TensorFlow 2.20.0
GPUs: 1
Compute dtype: float16

══ Stratified Resplit → data_resplit ══
  Class                    Total   Train   Valid    Test
  ──────────────────────────────────────────────────
  Acne                      1300     910     195     195
  Blackheads                 123      87      18      18
  Dark-Spots                 265     187      39      39
  Dry-Skin                   676     474     101     101
  Englarged-Pores             43      27       8       8
  Eyebags                    148     104      22      22
  Normal Skin                248     174      37      37
  Not-Skin                  1170     820     175     175
  Oily-Skin                  280     196      42      42
  Skin-Redness               157     111      23      23
  Whiteheads                 126      90      18      18
  Wrinkles                   311     219      46      46

Classes (12): ['Acne', 'Blackheads', 'Dark-Spots', 'Dry-Skin', 'Englarged-Pores', 'Eyebags', 'Normal Skin', 'Not-Sk

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                      ┃ Output Shape             ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)        │ (None, 260, 260, 3)      │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ resnet50v2 (Functional)           │ (None, 9, 9, 2048)       │    23,564,800 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ global_average_pooling2d          │ (None, 2048)             │             0 │
│ (GlobalAveragePooling2D)          │                          │               │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ batch_normalization               │ (None, 2048)             │         8,192 │
│ (BatchNormalization)              │                          │               │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dropout (Dropout)                 │ (None, 2048)             │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense (Dense)                     │ (None, 512)              │     1,049,088 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ batch_normalization_1             │ (None, 512)              │         2,048 │
│ (BatchNormalization)              │                          │               │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dropout_1 (Dropout)               │ (None, 512)              │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense_1 (Dense)                   │ (None, 256)              │       131,328 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ batch_normalization_2             │ (None, 256)              │         1,024 │
│ (BatchNormalization)              │                          │               │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dropout_2 (Dropout)               │ (None, 256)              │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense_2 (Dense)                   │ (None, 12)               │         3,084 │
└───────────────────────────────────┴──────────────────────────┴───────────────┘

 Total params: 24,759,564 (94.45 MB)

 Trainable params: 1,189,132 (4.54 MB)

 Non-trainable params: 23,570,432 (89.91 MB)


══ Phase 1: Head Warm-up ══
Epoch 1/12
107/107 ━━━━━━━━━━━━━━━━━━━━ 113s 652ms/step - accuracy: 0.4848 - loss: 2.0584 - val_accuracy: 0.7099 - val_loss: 1.2920 - learning_rate: 0.0030
Epoch 2/12
107/107 ━━━━━━━━━━━━━━━━━━━━ 34s 314ms/step - accuracy: 0.5846 - loss: 1.6682 - val_accuracy: 0.7224 - val_loss: 1.2112 - learning_rate: 0.0030
Epoch 3/12
107/107 ━━━━━━━━━━━━━━━━━━━━ 30s 282ms/step - accuracy: 0.6090 - loss: 1.5859 - val_accuracy: 0.7459 - val_loss: 1.1527 - learning_rate: 0.0030
Epoch 4/12
107/107 ━━━━━━━━━━━━━━━━━━━━ 30s 279ms/step - accuracy: 0.6314 - loss: 1.5240 - val_accuracy: 0.7528 - val_loss: 1.1221 - learning_rate: 0.0030
Epoch 5/12
107/107 ━━━━━━━━━━━━━━━━━━━━ 30s 275ms/step - accuracy: 0.6370 - loss: 1.5161 - val_accuracy: 0.7210 - val_loss: 1.1531 - learning_rate: 0.0030
Epoch 6/12
107/107 ━━━━━━━━━━━━━━━━━━━━ 31s 293ms/step - accuracy: 0.6487 - loss: 1.4900 - val_accuracy: 0.7735 - val_loss: 1.1000 - learning_rate: 0.0030
Epoch 7/12
107/107 ━━━━━━━━━━━━━━━━━━━━ 

In [9]:
print(model.output_shape)

(None, 12)


In [7]:
import os

classes = sorted(os.listdir(TRAIN_DIR))
print(classes)

['Acne', 'Blackheads', 'Dark-Spots', 'Dry-Skin', 'Englarged-Pores', 'Eyebags', 'Normal Skin', 'Not-Skin', 'Oily-Skin', 'Skin-Redness', 'Whiteheads', 'Wrinkles']


In [8]:
from pathlib import Path

classes = sorted([p.name for p in Path(TRAIN_DIR).iterdir() if p.is_dir()])
print(classes)

['Acne', 'Blackheads', 'Dark-Spots', 'Dry-Skin', 'Englarged-Pores', 'Eyebags', 'Normal Skin', 'Not-Skin', 'Oily-Skin', 'Skin-Redness', 'Whiteheads', 'Wrinkles']


In [14]:
from PIL import Image
import numpy as np
import tensorflow as tf

img_path = "./2EZ4GTUVFM17.jpg"

img = Image.open(img_path).convert("RGB").resize((260,260))
img = np.array(img, dtype=np.float32)
img = tf.keras.applications.resnet_v2.preprocess_input(img)

pred = model.predict(np.expand_dims(img,0), verbose=0)[0]

for i, p in enumerate(pred):
    print(f"{i:2d} {class_names[i]:15s} {p:.4f}")

print("\nPredicted:", class_names[np.argmax(pred)])

 0 Acne            0.0835
 1 Blackheads      0.0298
 2 Dark-Spots      0.0466
 3 Dry-Skin        0.0676
 4 Englarged-Pores 0.0180
 5 Eyebags         0.0260
 6 Normal Skin     0.0161
 7 Not-Skin        0.6105
 8 Oily-Skin       0.0178
 9 Skin-Redness    0.0152
10 Whiteheads      0.0419
11 Wrinkles        0.0270

Predicted: Not-Skin
